<a href="https://colab.research.google.com/github/muneer-ahmad10/Resume-Analyzer/blob/main/Resume_Analyzer_streamli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit pyngrok PyPDF2 sentence-transformers scikit-learn matplotlib spacy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 70.2 MB/s eta 0:00:00


In [11]:
%%writefile app.py

import streamlit as st
import re
import PyPDF2
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# ==================================================
# PAGE TITLE
# ==================================================

st.title("🚀 AI Resume Analyzer")

st.write("Upload Resume + Paste Job Description")

# ==================================================
# LOAD MODEL
# ==================================================

model = SentenceTransformer('all-MiniLM-L6-v2')

# ==================================================
# CLEAN TEXT
# ==================================================

def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z0-9 ]', ' ', text)

    text = " ".join(text.split())

    return text

# ==================================================
# EXTRACT PDF TEXT
# ==================================================

def extract_text_from_pdf(file):

    reader = PyPDF2.PdfReader(file)

    text = ""

    for page in reader.pages:

        extracted = page.extract_text()

        if extracted:
            text += extracted

    return text

# ==================================================
# SKILLS DATABASE
# ==================================================

SKILLS_DB = [

    "python",
    "machine learning",
    "deep learning",
    "nlp",
    "sql",
    "pytorch",
    "tensorflow",
    "streamlit",
    "rag",
    "azure",
    "mlops",
    "api",
    "apis",
    "vector databases",
    "semantic search",
    "document understanding",
    "llms",
    "data analysis",
    "pandas",
    "numpy",
    "scikit learn",
    "xgboost",
    "transformers"

]

recommendations = {
    "tensorflow":"Learn Deep Learning with TensorFlow",
    "pytorch":"Build Neural Networks using PyTorch",
    "rag":"Build a RAG chatbot project",
    "azure":"Learn Azure AI Services"
}

# ==================================================
# SKILL EXTRACTION
# ==================================================

def extract_skills(text):

    text = text.lower()

    found_skills = []

    for skill in SKILLS_DB:

        if skill in text:
            found_skills.append(skill)

    return list(set(found_skills))

# ==================================================
# SIMILARITY
# ==================================================

def compute_similarity(resume, job):

    emb1 = model.encode([resume])

    emb2 = model.encode([job])

    score = cosine_similarity(emb1, emb2)[0][0]

    return round(score * 100, 2)

def extract_email(text):

    text = text.replace(" ", "")

    pattern = r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}'

    emails = re.findall(pattern, text)

    if emails:
        return emails[0]

    return "Not Found"

def extract_phone(text):

    pattern = r'(\+?\d[\d\s\-]{8,15}\d)'

    phones = re.findall(pattern, text)

    if phones:
        return phones[0]

    return "Not Found"

def extract_name(text):

    lines = text.split("\n")

    for line in lines:

        line = line.strip()

        if len(line.split()) >= 2:

            if line.isupper():

                return line.title()

    return "Not Found"



# ==================================================
# USER INPUTS
# ==================================================

uploaded_file = st.file_uploader("Upload Resume PDF", type=["pdf"])

job_description = st.text_area("Paste Job Description")

# ==================================================
# MAIN LOGIC
# ==================================================

if uploaded_file and job_description:

    # Extract Resume Text
    resume_text = extract_text_from_pdf(uploaded_file)

    st.write("### Debug Resume Text")

    st.text(resume_text[:1000])

    name = extract_name(resume_text)

    email = extract_email(resume_text)

    phone = extract_phone(resume_text)

    # Clean
    resume_clean = clean_text(resume_text)

    job_clean = clean_text(job_description)

    # Skills
    job_skills = extract_skills(job_clean)

    resume_skills = extract_skills(resume_clean)

    matched = list(set(job_skills) & set(resume_skills))

    missing = list(set(job_skills) - set(resume_skills))

    # Scores
    semantic_score = compute_similarity(resume_clean, job_clean)

    skill_score = (
    len(matched) / len(job_skills) * 100
    if len(job_skills) > 0
    else 0
    )

    ats_score = round(
    semantic_score * 0.6 +
    skill_score * 0.4,
    2
    )

    # Verdict
    if semantic_score > 75:
        verdict = "Strong Match ✅"

    elif semantic_score > 50:
        verdict = "Moderate Match ⚠️"

    else:
        verdict = "Low Match ❌"

    strengths = []

    if "python" in matched:
        strengths.append("Python expertise detected")

    if "machine learning" in matched:
        strengths.append("Machine Learning experience detected")

    if "nlp" in matched:
        strengths.append("NLP skills detected")

    if semantic_score > 50:
        strengths.append("Resume aligns reasonably with job description")


    improvements = []

    if "tensorflow" in missing:
        improvements.append(
            "Add TensorFlow projects to strengthen deep learning profile."
        )

    if "mlops" in missing:
        improvements.append(
            "Learn deployment and MLOps workflows."
        )

    if semantic_score < 70:
        improvements.append(
            "Tailor your resume keywords according to the job description."
        )

    # ==================================================
    # DISPLAY RESULTS
    # ==================================================

    st.subheader("📊 Analysis Results")

    st.subheader("📄 Resume Information")

    st.write(f"**Name:** {name}")

    st.write(f"**Email:** {email}")

    st.write(f"**Phone:** {phone}")

    st.write(f"### Semantic Match Score: {semantic_score}%")

    if semantic_score < 70:

      st.warning(
          "Your resume may not align strongly with this job description."
      )

    st.metric(
    "ATS Compatibility Score",
    f"{ats_score}%"
    )

    st.write(f"### Skill Match Score: {round(skill_score,2)}%")

    st.write(f"### Verdict: {verdict}")

    st.write("## ✅ Matched Skills")

    st.write(matched)

    st.write("## ❌ Missing Skills")

    st.write(missing)

    st.subheader("📚 Learning Recommendations")

    for skill in missing:

      if skill in recommendations:

          st.write(
              f"✅ {skill} → {recommendations[skill]}"
          )

    st.subheader("💪 Resume Strengths")

    for s in strengths:
       st.success(s)

    st.subheader("🚀 Resume Improvement Suggestions")

    for item in improvements:
        st.warning(item)



    # ==================================================
    # CHART
    # ==================================================

    fig, ax = plt.subplots()

    labels = ['Matched', 'Missing']

    counts = [len(matched), len(missing)]

    ax.bar(labels, counts)

    st.pyplot(fig)

Overwriting app.py


In [ ]:
# !streamlit run app.py & npx localtunnel --port 8501

from pyngrok import ngrok

# 1. Plug in your token here
ngrok.set_auth_token("3DQMvg7DN2UTMoMvgHFxvhcT66M_7v4cXJhmhSmNNaXvK447J")

# 2. Open the tunnel
public_url = ngrok.connect(8501)
print(f"Click here to open your app: {public_url}")

# 3. Run Streamlit
!streamlit run app.py

Click here to open your app: NgrokTunnel: "https://handbook-marry-hazy.ngrok-free.dev" -> "http://localhost:8501"




2026-06-07 03:21:43.423 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.46.105:8501

Loading weights: 100% 103/103 [00:00<00:00, 4230.41it/s]
[transformers] Accessing `__path__` from `.models.aria.image_processing_aria`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `__path__` from `.models.aria.image_processing_pil_aria`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `__path__` from `.models.auto.image_processing_auto`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `__path__` from `.models.beit.image_processing_beit`. Returning `__path__` instead. Behavior may be different and this 